# Week 3: Transfer Learning — mT5 Fine-Tuned for Dholuo

**Trizah — mT5-small, English/Kiswahili → Dholuo**

## Objective
Fine-tune `google/mt5-small` to translate Public Service Announcement (PSA) content
from **English** and **Kiswahili** into **Dholuo**.

## Language adaptation approach
mT5 has no per-language embedding table (unlike NLLB), so there is no language token to
add or resize. Since mT5 was pretrained on mC4, which does **not** include Dholuo, this is
treated as **partial support**: the model must learn the language largely from this
fine-tuning data, guided by a **consistent text-prefix scheme** rather than a language
embedding. Every source sentence is prefixed with an explicit natural-language instruction
(`"translate English to Dholuo: "` / `"tafsiri Kiswahili kwa Dholuo: "`) so the model has a
constant, learnable signal for which task it is performing — this is the standard mT5/T5
mechanism for multi-task conditioning in the absence of dedicated language tokens.

This notebook picks up directly from `dholuo_somali_preprocessing_eda.ipynb`, which produced
the cleaned dataset used below.

## Step 1: Install Required Libraries

In [1]:
#
# pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate


^C


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\DELL\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\torch\\include\\torch\\csrc\\api\\include\\torch\\data\\samplers\\base.h'
Check the permissions.


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Step 3: Mount Google Drive

Required so checkpoints and the final model survive a Colab runtime disconnect
(as happened during Week 2). Point `DRIVE_PROJECT_DIR` at wherever you keep the
project in your own Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/psa-dholuo-mt"   # <-- adjust to your Drive path
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints/mt5-dholuo"
MODEL_OUT_DIR = f"{DRIVE_PROJECT_DIR}/models/mt5-dholuo-final"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODEL_OUT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)
print("Final model will be saved to:", MODEL_OUT_DIR)


## Step 4: Load the Cleaned Dataset

Loads the output of `dholuo_somali_preprocessing_eda.ipynb`
(`psa_dataset_dholuo_somali_cleaned.csv`, 16,029 rows). Only the columns needed
for the Dholuo track are kept — `Somali` is dropped (that's Patricia's target
language, not part of this notebook).

In [ ]:
DATA_PATH = f"{DRIVE_PROJECT_DIR}/data/processed/psa_dataset_dholuo_somali_cleaned.csv"
# If you'd rather upload directly instead of reading from Drive, comment the line above and
# uncomment the block below:
# from google.colab import files
# uploaded = files.upload()
# DATA_PATH = list(uploaded.keys())[0]

df = pd.read_csv(DATA_PATH)
df = df[["PSA_Id", "Domain", "English", "Kiswahili", "Dholuo", "Class", "Source"]].copy()

print("Dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
df.head()


## Step 5: Build Translation Pairs (with Dholuo prefix scheme)

Each row becomes **two** training examples: English→Dholuo and Kiswahili→Dholuo.
The explicit prefix is the language-adaptation workaround for mT5 (see intro cell).

In [ ]:
ENG_PREFIX = "translate English to Dholuo: "
SWH_PREFIX = "tafsiri Kiswahili kwa Dholuo: "

translation_pairs = []

for _, row in df.iterrows():
    translation_pairs.append({
        "source_text": ENG_PREFIX + str(row["English"]),
        "target_text": str(row["Dholuo"]),
        "source_lang": "eng",
        "target_lang": "luo",
        "domain": row["Domain"],
    })
    translation_pairs.append({
        "source_text": SWH_PREFIX + str(row["Kiswahili"]),
        "target_text": str(row["Dholuo"]),
        "source_lang": "swh",
        "target_lang": "luo",
        "domain": row["Domain"],
    })

translation_df = pd.DataFrame(translation_pairs)
translation_df["translation_direction"] = translation_df["source_lang"].map({
    "eng": "eng-luo",
    "swh": "swh-luo",
})

print("Translation pairs shape:", translation_df.shape)
translation_df.head()


## Step 6: Quality Checks

Same checks the team applied for Ekegusii: missing values, empty strings, exact
duplicate pairs.

In [ ]:
print("Missing values:\n", translation_df.isnull().sum())

empty_src = (translation_df["source_text"].str.strip() == "").sum()
empty_tgt = (translation_df["target_text"].str.strip() == "").sum()
print(f"\nEmpty source rows: {empty_src}")
print(f"Empty target rows: {empty_tgt}")

dupes = translation_df.duplicated(subset=["source_text", "target_text"]).sum()
print(f"Exact duplicate pairs: {dupes}")

translation_df = translation_df.drop_duplicates(subset=["source_text", "target_text"]).reset_index(drop=True)
print(f"\nFinal translation pair count after dedup: {len(translation_df)}")


## Step 7: Stratified Train / Validation / Test Split

80/10/10 split, stratified by `source_lang` so English and Kiswahili stay
proportionally represented in every split (same approach the team used for
Ekegusii).

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    translation_df,
    test_size=0.20,
    random_state=42,
    stratify=translation_df["source_lang"],
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["source_lang"],
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
validation_dataset = Dataset.from_pandas(validation_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print("Training examples  :", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("Test examples       :", len(test_dataset))

from collections import Counter
print("\nTrain source_lang distribution:", Counter(train_dataset["source_lang"]))
print("Val source_lang distribution  :", Counter(validation_dataset["source_lang"]))
print("Test source_lang distribution :", Counter(test_dataset["source_lang"]))


## Step 8: Load Tokenizer and Model

In [ ]:
MT5_MODEL = "google/mt5-small"

print(f"Loading tokenizer: {MT5_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(MT5_MODEL)
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

print(f"\nLoading model: {MT5_MODEL}")
model = AutoModelForSeq2SeqLM.from_pretrained(MT5_MODEL)

if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id
    print(f"Set decoder_start_token_id to {tokenizer.pad_token_id}")

print(f"Model parameters: {model.num_parameters():,}")


## Step 9: Tokenize Datasets

mT5 uses SentencePiece with no dedicated language tokens, so tokenization is
plain text-in/text-out — the prefix added in Step 5 is what carries the task
signal.

In [ ]:
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 256

def tokenize_function(example):
    model_inputs = tokenizer(
        example["source_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        example["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing training set...")
tokenized_train = train_dataset.map(tokenize_function, remove_columns=train_dataset.column_names)
print("Tokenizing validation set...")
tokenized_val = validation_dataset.map(tokenize_function, remove_columns=validation_dataset.column_names)
print("Tokenizing test set...")
tokenized_test = test_dataset.map(tokenize_function, remove_columns=test_dataset.column_names)

print("\nSample tokenization:")
sample = train_dataset[0]
tok_sample = tokenize_function(sample)
print("Source:", sample["source_text"][:80], "...")
print("Source tokens:", len(tok_sample["input_ids"]))
print("Target tokens:", len(tok_sample["labels"]))


## Step 10: Data Collator

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=tokenizer.pad_token_id,
)
print("Data collator ready.")


## Step 11: Evaluation Metrics

BLEU, SacreBLEU, and chrF — matching the metrics used across the whole team's
models for a consistent, comparable Week 3 report.

In [ ]:
bleu_metric = evaluate.load("bleu")
sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels_bleu = [[l.strip()] for l in decoded_labels]

    try:
        bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels_bleu)["bleu"]
    except Exception as e:
        print("BLEU failed:", e); bleu = 0.0

    try:
        sacrebleu = sacrebleu_metric.compute(predictions=decoded_preds, references=decoded_labels_bleu)["score"]
    except Exception as e:
        print("SacreBLEU failed:", e); sacrebleu = 0.0

    try:
        chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels_bleu)["score"]
    except Exception as e:
        print("chrF failed:", e); chrf = 0.0

    return {"bleu": bleu, "sacrebleu": sacrebleu, "chrf": chrf}

print("Metrics loaded: BLEU, SacreBLEU, chrF")


## Step 12: Training Arguments

Per the Week 3 requirements: **10 epochs**, `save_strategy="epoch"` (automatic
checkpointing, no manual saves needed), checkpoints written to the mounted
Drive so nothing is lost on a runtime disconnect.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=10,
    predict_with_generate=True,
    generation_max_length=256,
    generation_num_beams=4,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="sacrebleu",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    save_total_limit=3,
    warmup_steps=100,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer ready.")
print(f"Training examples  : {len(tokenized_train):,}")
print(f"Validation examples: {len(tokenized_val):,}")
print(f"Epochs: 10 | Batch size: 8 | Checkpoints: {CHECKPOINT_DIR}")


## Step 13: Train

In [ ]:
torch.cuda.empty_cache()
gc.collect()

train_result = trainer.train()

print("Final training loss:", train_result.training_loss)
print("Training time (min):", train_result.metrics.get("train_runtime", 0) / 60)


## Step 14: Per-Epoch Results Table

Pulls the epoch-by-epoch training/validation loss and metrics straight from
the trainer's log history — this is the table for **Report Section 3
(Results)**.

In [ ]:
log_history = trainer.state.log_history

epoch_rows = []
for entry in log_history:
    if "eval_loss" in entry:
        epoch_rows.append({
            "Epoch": entry.get("epoch"),
            "Training Loss": None,  # filled in below from the preceding 'loss' entry
            "Validation Loss": entry.get("eval_loss"),
            "BLEU": entry.get("eval_bleu"),
            "SacreBLEU": entry.get("eval_sacrebleu"),
            "chrF": entry.get("eval_chrf"),
        })

train_losses = [e["loss"] for e in log_history if "loss" in e and "eval_loss" not in e]
for i, row in enumerate(epoch_rows):
    if i < len(train_losses):
        row["Training Loss"] = train_losses[min(i, len(train_losses)-1)]

results_df = pd.DataFrame(epoch_rows)
print(results_df.to_string(index=False))

best_epoch_idx = results_df["SacreBLEU"].idxmax()
print(f"\nBest epoch by SacreBLEU: epoch {results_df.loc[best_epoch_idx, 'Epoch']}")
print(f"(metric_for_best_model='sacrebleu', greater_is_better=True)")


## Step 15: Sample Translations

5–10 example comparisons for **Report Section 5**.

In [ ]:
import random
random.seed(42)

sample_idx = random.sample(range(len(test_dataset)), 10)

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

rows = []
for i in sample_idx:
    example = test_dataset[i]
    inputs = tokenizer(example["source_text"], return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH).to(device)
    with torch.no_grad():
        generated = model.generate(**inputs, max_length=MAX_TARGET_LENGTH, num_beams=4)
    prediction = tokenizer.decode(generated[0], skip_special_tokens=True)

    rows.append({
        "Source": example["source_text"],
        "Reference": example["target_text"],
        "Prediction": prediction,
        "Direction": example["translation_direction"],
    })

samples_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 100)
samples_df


## Step 16: Final Test-Set Evaluation

In [ ]:
test_results = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")
print("Test set results:")
for k, v in test_results.items():
    print(f"  {k}: {v}")


## Step 17: Save Final Model

Saved to Google Drive (per the mandatory Drive-mount requirement), plus a
zipped copy for submission.

In [ ]:
trainer.save_model(MODEL_OUT_DIR)
tokenizer.save_pretrained(MODEL_OUT_DIR)
print(f"Model saved to {MODEL_OUT_DIR}")

import shutil
zip_path = f"{DRIVE_PROJECT_DIR}/models/mt5-dholuo-final"
shutil.make_archive(zip_path, "zip", MODEL_OUT_DIR)
print(f"Zipped model available at {zip_path}.zip")


---
## Report Scaffold (fill in after training completes)

Use this section as the basis for `Trizah — mT5 Fine-Tuned for Dholuo` (the
Week 3 submission report).

### 1. Setup
- **Base model:** `google/mt5-small`
- **Why selected:** assigned model for this track; also a natural comparison
  point against Steve's NLLB-Dholuo run since both target the same language.
- **Language support:** Dholuo is **not natively supported** by mT5 (not in
  its mC4 pretraining data). mT5 has no per-language embedding table to
  extend, so the workaround used here is a **consistent text-prefix scheme**
  (`"translate English to Dholuo: "` / `"tafsiri Kiswahili kwa Dholuo: "`)
  rather than an embedding modification — see the intro cell for the full
  rationale.

### 2. Dataset
- Source: `data/processed/psa_dataset_dholuo_somali_cleaned.csv` (16,029 base
  rows → run Step 6 for the exact post-dedup pair count).
- Split: 80/10/10, stratified by source language — see Step 7 output for exact
  counts.
- Preprocessing carried over from `dholuo_somali_preprocessing_eda.ipynb`:
  curly-quote normalization, embedded-caption stripping, mojibake correction.
  Known limitation carried over: Dholuo translation quality has no automated
  validation tool (no `langdetect`/fastText coverage) — depends on Rencia's
  native-speaker QA pass.

### 3. Results
*(paste the Step 14 table here once training finishes)*
- Best epoch: *(from Step 14 output)*
- Selection metric: SacreBLEU (`metric_for_best_model="sacrebleu"`), chosen
  over raw BLEU because SacreBLEU is corpus-level, tokenization-standardized,
  and directly comparable across the team's different models.

### 5. Sample Translations
*(paste the Step 15 table here)*

### 6. Challenges Faced
- Dataset size: ~32k translation pairs (16k rows × 2 source languages) means
  10 full epochs on mT5-small is a meaningful Colab time commitment — note
  actual wall-clock time from Step 13 here.
- mT5 has no native Dholuo support, so early-epoch outputs may be close to
  copying the source or degenerate; note whether this resolved by later
  epochs.
- *(add anything Colab-specific: disconnects, OOM, etc.)*

### 7. Limitations
- A large share of the underlying English/Kiswahili source sentences are
  synthetically generated (fact-grounded, not organic PSA text) — noted
  transparently in Week 1; this may affect how naturally the model's Dholuo
  output reads on real-world PSA phrasing.
- No automated Dholuo QA tool exists, so reported translation quality is
  bounded by manual review coverage.
- Potential improvements: more epochs, larger mT5 checkpoint (base vs small),
  incorporating the glossary (`psa_glossary_dholuo_somali.csv`) to enforce
  consistent handling of institution names/acronyms during generation.
